- this notebook is just for testing purposes, it tests whether the packages work together

#### Imports

In [2]:
from unsloth import FastLanguageModel
import torch
import os 

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import sys
from unittest.mock import MagicMock

sys.modules['mergekit'] = MagicMock()
sys.modules['mergekit.config'] = MagicMock()
sys.modules['mergekit.merge'] = MagicMock()
sys.modules['llm_blender'] = MagicMock()
sys.modules['weave'] = MagicMock()
sys.modules['weave.trace'] = MagicMock()
sys.modules['weave.trace.context'] = MagicMock()

from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported
import torch
import re

In [5]:
import warnings
warnings.filterwarnings("ignore")

#### Load the dataset

In [14]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive")

splits = [
    dataset["level_1"].filter(lambda x: x["Correct"] == True),
    dataset["level_2"].filter(lambda x: x["Correct"] == True),
    dataset["level_3"].filter(lambda x: x["Correct"] == True),
]

dataset = concatenate_datasets(splits)


# pick 20 samples to do a health check on the pipeline 
dataset = dataset.select(range(20))

print(f"Training on {len(dataset)} samples")

Training on 20 samples


In [15]:
print(dataset.column_names)

['Op_Name', 'Level_ID', 'Task_ID', 'Kernel_Name', 'CUDA_Runtime', 'PyTorch_Native_Runtime', 'PyTorch_Compile_Runtime', 'CUDA_Speedup_Native', 'CUDA_Speedup_Compile', 'CUDA_Code', 'PyTorch_Code_Module', 'PyTorch_Code_Functional', 'Correct', 'Max_Diff', 'Error', 'NCU_Profile', 'Torch_Profile', 'Clang_Tidy', '__index_level_0__']


#### format data for GRPO

In [16]:
def format_for_grpo(example):
    prompt = (
        f"Optimize the following CUDA kernel for maximum performance while maintaining correctness.\n\n"
        f"```cuda\n{example['CUDA_Code']}\n```\n\n"
        f"Return ONLY the optimized CUDA code. Do not include explanations, markdown, or extra text."
    )
    return {
        "prompt": prompt,
        "cuda_runtime": example["CUDA_Runtime"],
        "speedup_native": example["CUDA_Speedup_Native"],
        "speedup_compile": example["CUDA_Speedup_Compile"]
    }

dataset = dataset.map(format_for_grpo)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

#### reward function

In [17]:
def cuda_optimization_reward(completions, cuda_runtime=None, speedup_native=None, speedup_compile=None, **kwargs):
    rewards = []
    n = len(completions)
    
    cuda_rt = cuda_runtime if cuda_runtime else [0.0] * n
    sp_nat = speedup_native if speedup_native else [1.0] * n
    sp_com = speedup_compile if speedup_compile else [1.0] * n
    
    for i, comp in enumerate(completions):
        base_reward = 0.6 * sp_nat[i] + 0.4 * sp_com[i]        
        ## afterwards we can edit in this
        rewards.append(base_reward)
        
    return rewards

#### load the model

In [18]:
max_seq_length = 2048   
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None, 
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                         
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,               
    bias="none",               
    use_gradient_checkpointing="unsloth",  
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


#### GRPO configuration

In [19]:
training_args = GRPOConfig(
    output_dir="./qwen25_cuda_grpo",
    run_name="cuda_opt_grpo_v1",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_prompt_length=1024,
    max_completion_length=1024,
    num_generations=4,
    temperature=0.7,
    top_p=0.9,
    logging_steps=1,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    report_to="tensorboard",
    beta=0.04,
    epsilon=0.2,
)

In [20]:
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}
    
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=cuda_optimization_reward,
    args=training_args,
    train_dataset=dataset,
)

I0000 00:00:1776017141.023386    3506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776017141.065320    3506 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


#### training

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will